# GT Tools - Cek Alignment Frame & Pengukuran 3x



**PENTING:** GT dan FRAMES_CSV harus dari **video yang sama**. Notebook komparasi hanya menyimpan GT satu video pada satu waktu, jadi pastikan pasangannya benar. Bagian 1 otomatis memperingatkan kalau salah pasang.


## Bagian 1 - Cek Alignment Frame GT

Untuk tiap frame GT, dilihat error sudut model di frame itu dan di frame tetangga (offset -3..+3).

- Kalau rata-rata error kecil dan best_offset kebanyakan 0: alignment wajar.
- Kalau rata-rata error besar (puluhan derajat) dan best_offset acak/menempel di tepi window: **GT kemungkinan salah pasang video**, atau ada pergeseran sistematis.

Catatan: di fase gerakan cepat satu frame tetangga kadang kebetulan lebih cocok; baca POLA keseluruhan, bukan satu baris.


In [ ]:
import pandas as pd, numpy as np, collections

# ====== KONFIG (pastikan GT dan CSV dari video yang SAMA) ======
FRAMES_CSV = r"C:\Users\ASUS\OneDrive\Documents\Claude\Projects\Seputar Tugas Akhir\Dataset video Bicep curl\comparison_outputs_v3\output subjek 2\frames_blazepose.csv"
ANGLE_COL  = "angle_smooth"
WINDOW     = 3

# GT untuk video di atas (contoh ini = subjek 2). Ganti sesuai videonya.
GROUND_TRUTH_ANGLES = [
    ( 69, 157.924),( 85,  97.839),( 90,  68.149),(100,  36.018),(131,  96.785),
    (152, 165.717),(256,  57.593),(333,  74.466),(344,  40.779),(454, 105.033),
    (470, 142.360),(492,  56.961),(566,  76.298),(579,  35.000),(608, 104.732),
]

df = pd.read_csv(FRAMES_CSV)
ang = dict(zip(df["frame"], df[ANGLE_COL]))
rows=[]
for f, gt in GROUND_TRUTH_ANGLES:
    a0 = ang.get(f, np.nan)
    err0 = abs(a0-gt) if (a0==a0 and a0!=0) else np.nan
    best_off, best_err = 0, err0
    for off in range(-WINDOW, WINDOW+1):
        a = ang.get(f+off, np.nan)
        if a!=a or a==0: continue
        e = abs(a-gt)
        if best_err!=best_err or e < best_err: best_err, best_off = e, off
    rows.append({'frame':f,'gt':gt,'model_frame':round(a0,2) if a0==a0 else None,
                 'err_frame':round(err0,2) if err0==err0 else None,
                 'best_offset':best_off,'err_best':round(best_err,2) if best_err==best_err else None})
res=pd.DataFrame(rows)
print(res.to_string(index=False))
cnt=collections.Counter(r['best_offset'] for r in rows)
print("\nDistribusi best_offset:", dict(sorted(cnt.items())))
print(f"Frame sudah pas (offset 0): {cnt.get(0,0)} dari {len(rows)}")
mean_err0=np.nanmean([r['err_frame'] for r in rows])
print(f"Rata-rata error di frame GT: {mean_err0:.2f} deg")
mode_off=cnt.most_common(1)[0][0]
if mean_err0 > 25:
    print("\nPERINGATAN: rata-rata error sangat besar. GT kemungkinan SALAH PASANG video. Cek FRAMES_CSV vs GROUND_TRUTH_ANGLES dari video yang sama.")
elif mode_off!=0 and cnt.get(mode_off,0) > len(rows)*0.5:
    print(f"\nPERINGATAN: mayoritas cocok di offset {mode_off:+d}. Indikasi pergeseran sistematis {abs(mode_off)} frame. Cek penomoran ImageJ vs kode.")
else:
    print("\nAlignment GT terlihat wajar (tidak ada pergeseran sistematis).")


## Bagian 2 - Template Pengukuran GT 3x + Reliabilitas

Isi `measurements`: per frame, fase, dan **3 hasil ukur ImageJ**. Output: rata-rata + selisih (range) + std per frame, ringkasan reliabilitas, dan list GROUND_TRUTH_ANGLES baru siap tempel.

Angka di bawah masih placeholder (nilai lama diulang 3x). GANTI dengan 3 hasil ukurmu, mis. [89, 90, 92].


In [ ]:
import pandas as pd, numpy as np, os

# ====== ISI: frame -> (fase, [ukur1, ukur2, ukur3]) ======
measurements = {
    69:  ('ekstensi',   [157.9, 157.9, 157.9]),
    85:  ('konsentrik', [97.8, 97.8, 97.8]),
    90:  ('mid curl',   [68.1, 68.1, 68.1]),
    100: ('fleksi',     [36.0, 36.0, 36.0]),
    131: ('eksentrik',  [96.8, 96.8, 96.8]),
    152: ('ekstensi',   [165.7, 165.7, 165.7]),
    256: ('konsentrik', [57.6, 57.6, 57.6]),
    333: ('mid curl',   [74.5, 74.5, 74.5]),
    344: ('fleksi',     [40.8, 40.8, 40.8]),
    454: ('eksentrik',  [105.0, 105.0, 105.0]),
    470: ('ekstensi',   [142.4, 142.4, 142.4]),
    492: ('konsentrik', [57.0, 57.0, 57.0]),
    566: ('mid curl',   [76.3, 76.3, 76.3]),
    579: ('fleksi',     [35.0, 35.0, 35.0]),
    608: ('eksentrik',  [104.7, 104.7, 104.7]),
}
OUT_DIR = '.'
rows=[]
for f,(fase,m) in measurements.items():
    m=np.array(m, float)
    rows.append({'frame':f,'fase':fase,'ukur1':m[0],'ukur2':m[1],'ukur3':m[2],
                 'mean':round(m.mean(),2),'range':round(m.max()-m.min(),2),'std':round(m.std(ddof=1),2)})
gt=pd.DataFrame(rows).sort_values('frame')
print(gt.to_string(index=False))
print("\n=== Reliabilitas Ground Truth ===")
print(f"Rata-rata selisih antar pengukuran (range): {gt['range'].mean():.2f} deg")
imax=gt['range'].idxmax()
print(f"Selisih terbesar: {gt.loc[imax,'range']:.2f} deg di frame {int(gt.loc[imax,'frame'])} ({gt.loc[imax,'fase']})")
print(f"Rata-rata std: {gt['std'].mean():.2f} deg")
print(f"Frame dengan range <= 3 deg (konsisten): {(gt['range']<=3).sum()} dari {len(gt)}")
p=os.path.join(OUT_DIR,'GT_3x_reliabilitas.csv'); gt.to_csv(p, index=False, encoding='utf-8-sig')
print(f'\nTersimpan: {p}')
print("\n=== GROUND_TRUTH_ANGLES baru (tempel ke Cell 4 notebook komparasi) ===")
print('GROUND_TRUTH_ANGLES = [')
for _,r in gt.iterrows():
    print(f"    ({int(r['frame']):4d}, {r['mean']:8.3f}),  # {r['fase']}")
print(']')
